# AgroPredict Colombia
## Optimización y evaluación de modelos predictivos con rezagos t-1 y t-2

**Despliegue de Soluciones Analíticas — Tercera entrega**

### Objetivo

Optimizar y comparar los modelos supervisados desarrollados en
`04.1_modelado_2periodos.ipynb` para estimar el rendimiento municipal del maíz en
Colombia (t/ha), usando como variables el rendimiento del semestre anterior (t-1),
el del mismo semestre del año anterior (t-2), el semestre, el departamento y el
sistema productivo, sin el año.

En la comparación previa, con hiperparámetros fijos, XGBoost obtuvo el menor RMSE
(0.519 t/ha en validación 2024), seguido de cerca por Random Forest (0.530 t/ha),
mientras que Ridge quedó por detrás (0.543 t/ha). Por eso este cuaderno optimiza
Random Forest y XGBoost.

### Metodología

1. Construir el modelo de referencia (persistencia) basado en el rendimiento del
   semestre anterior.
2. Implementar validación temporal con ventanas expansivas.
3. Optimizar hiperparámetros de Random Forest y XGBoost.
4. Comparar el desempeño mediante RMSE (métrica principal), MAE y R².
5. Evaluar el modelo seleccionado sobre 2025 y reentrenarlo con 2019–2025 para
   producción.
6. Registrar los experimentos en MLflow.

### Consideraciones metodológicas

- Se usan solo las filas con ambos rezagos disponibles. Como 2019 no tiene t-2, el
  entrenamiento efectivo empieza en 2020, y las ventanas de validación cruzada son:
  2020 → 2021, 2020–2021 → 2022 y 2020–2022 → 2023.
- Selección de hiperparámetros: promedio del RMSE en las tres ventanas.
- Validación: año 2024, para comparar los modelos optimizados.
- Prueba final: año 2025, reservado hasta seleccionar el modelo y evaluado una sola
  vez.
- Modelo de producción: el seleccionado, reentrenado con 2019–2025 con los mismos
  hiperparámetros y sin ajustes adicionales.
- Se usa el RMSE como métrica principal porque en cerca del 42 % de los registros el
  rendimiento se repite exactamente del semestre anterior, lo que favorece
  artificialmente el MAE de la persistencia.
- Se utilizan únicamente variables disponibles antes de la siembra y se conserva la
  separación temporal para evitar fuga de información.

In [2]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import mlflow

from dotenv import load_dotenv

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor


# Raíz del repositorio: se busca hacia arriba la carpeta que contiene .git
def encontrar_raiz(inicio=Path.cwd()):
    for carpeta in [inicio.resolve(), *inicio.resolve().parents]:
        if (carpeta / ".git").exists():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz del repositorio (.git)")


# Rutas del proyecto
PROYECTO = encontrar_raiz()
DATOS = PROYECTO / "data" / "processed" / "features_rezago_1_2.parquet"

# Configuración de MLflow
load_dotenv(PROYECTO / "config" / ".env", override=True)

MLFLOW_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_EXPERIMENTO = os.getenv("MLFLOW_EXPERIMENT")

mlflow.set_tracking_uri(MLFLOW_URI)
experimento = mlflow.set_experiment(MLFLOW_EXPERIMENTO)

# Etiquetas comunes para todos los runs del cuaderno
TAGS_COMUNES = {
    "notebook": "05.1_optimizacion_modelos",
    "variables_rezago": "t-1_t-2",
}

print("Raíz del proyecto:", PROYECTO)
print("Archivo de datos:", DATOS)
print("Datos disponibles:", DATOS.exists())
print("Servidor MLflow:", MLFLOW_URI)
print("Experimento:", MLFLOW_EXPERIMENTO, "| id:", experimento.experiment_id)

Raíz del proyecto: /Users/Simon/Library/CloudStorage/OneDrive-SharedLibraries-ProductosAlimenticiosPascualS.A/Inteligencia Comercial - 5-Semestre/MIID-4304 Despliegue/Proyecto/agropredict-colombia
Archivo de datos: /Users/Simon/Library/CloudStorage/OneDrive-SharedLibraries-ProductosAlimenticiosPascualS.A/Inteligencia Comercial - 5-Semestre/MIID-4304 Despliegue/Proyecto/agropredict-colombia/data/processed/features_rezago_1_2.parquet
Datos disponibles: True
Servidor MLflow: http://13.221.115.228:5000
Experimento: agropredict-maiz-rezagos-t1-t2 | id: 2


In [3]:
# Cargar datos con ambos rezagos
df = pd.read_parquet(DATOS)

VARIABLES = [
    "semestre_b",
    "rendimiento_rezago_1",
    "rendimiento_rezago_2",
    "departamento",
    "sistema_productivo"
]

OBJETIVO = "rendimiento"

# Conservar observaciones completas para el modelado
df_modelo = df.dropna(
    subset=VARIABLES + [OBJETIVO]
).copy()

# Separación temporal
train = df_modelo[df_modelo["anio"].between(2020, 2023)].copy()
val = df_modelo[df_modelo["anio"] == 2024].copy()
test = df_modelo[df_modelo["anio"] == 2025].copy()

print("=== DISTRIBUCIÓN TEMPORAL ===")
print(f"Datos originales: {len(df):,}")
print(f"Datos utilizables: {len(df_modelo):,}")
print(f"Entrenamiento 2020–2023: {len(train):,}")
print(f"Validación 2024: {len(val):,}")
print(f"Prueba final 2025: {len(test):,}")

print("\n=== OBSERVACIONES POR AÑO ===")
print(df_modelo.groupby("anio").size())

print("\n=== VARIABLES DEL MODELO ===")
print(VARIABLES)

=== DISTRIBUCIÓN TEMPORAL ===
Datos originales: 14,935
Datos utilizables: 10,782
Entrenamiento 2020–2023: 7,334
Validación 2024: 1,713
Prueba final 2025: 1,735

=== OBSERVACIONES POR AÑO ===
anio
2020    1839
2021    1916
2022    1810
2023    1769
2024    1713
2025    1735
dtype: int64

=== VARIABLES DEL MODELO ===
['semestre_b', 'rendimiento_rezago_1', 'rendimiento_rezago_2', 'departamento', 'sistema_productivo']


In [4]:
# Ventanas expansivas para validación cruzada temporal
VENTANAS_CV = [
    ([2020], 2021),
    ([2020, 2021], 2022),
    ([2020, 2021, 2022], 2023),
]

def calcular_metricas(y_real, y_pred):
    return {
        "rmse": np.sqrt(mean_squared_error(y_real, y_pred)),
        "mae": mean_absolute_error(y_real, y_pred),
        "r2": r2_score(y_real, y_pred)
    }

resultados_baseline = []

for anios_entrenamiento, anio_validacion in VENTANAS_CV:
    datos_validacion = train[
        train["anio"] == anio_validacion
    ]

    metricas = calcular_metricas(
        datos_validacion[OBJETIVO],
        datos_validacion["rendimiento_rezago_1"]
    )

    resultados_baseline.append({
        "ventana": f"{min(anios_entrenamiento)}-{max(anios_entrenamiento)} → {anio_validacion}",
        "anio_validacion": anio_validacion,
        "n_validacion": len(datos_validacion),
        **metricas
    })

df_baseline = pd.DataFrame(resultados_baseline)
promedio_baseline = df_baseline[["rmse", "mae", "r2"]].mean()

# Registro del baseline en MLflow
with mlflow.start_run(run_name="Baseline_persistencia_t1", tags=TAGS_COMUNES):

    mlflow.log_params({
        "modelo": "Persistencia_t1",
        "regla": "rendimiento_t = rendimiento_rezago_1",
        "tipo_validacion": "expanding_window",
        "numero_folds": len(VENTANAS_CV),
    })

    mlflow.log_metrics({
        "cv_rmse": float(promedio_baseline["rmse"]),
        "cv_mae": float(promedio_baseline["mae"]),
        "cv_r2": float(promedio_baseline["r2"]),
    })

    for _, fila in df_baseline.iterrows():
        anio = int(fila["anio_validacion"])
        mlflow.log_metrics({
            f"rmse_{anio}": float(fila["rmse"]),
            f"mae_{anio}": float(fila["mae"]),
            f"r2_{anio}": float(fila["r2"]),
        })

    mlflow.log_table(df_baseline, artifact_file="baseline_por_ventana.json")

print("=== BASELINE DE PERSISTENCIA POR VENTANA ===")
display(df_baseline.round(4))

print("\n=== PROMEDIO DE MÉTRICAS ===")
display(promedio_baseline.round(4))
print("\nBaseline registrado en MLflow.")

🏃 View run Baseline_persistencia_t1 at: http://13.221.115.228:5000/#/experiments/2/runs/c5d58c56e9ec457ba49ab104ae7969ef
🧪 View experiment at: http://13.221.115.228:5000/#/experiments/2
=== BASELINE DE PERSISTENCIA POR VENTANA ===


,ventana,anio_validacion,n_validacion,rmse,mae,r2
0,2020-2020 → 2021,2021,1916,1.6935,0.7763,-0.3576
1,2020-2021 → 2022,2022,1810,0.6633,0.3052,0.7784
2,2020-2022 → 2023,2023,1769,0.4985,0.1978,0.8787



=== PROMEDIO DE MÉTRICAS ===


rmse    0.9518
mae     0.4264
r2      0.4332
dtype: float64


Baseline registrado en MLflow.


In [5]:
# Variables predictoras y objetivo
X_train = train[VARIABLES]
y_train = train[OBJETIVO]

X_val = val[VARIABLES]
y_val = val[OBJETIVO]

# Clasificación de variables
NUMERICAS = [
    "semestre_b",
    "rendimiento_rezago_1",
    "rendimiento_rezago_2"
]

CATEGORICAS = [
    "departamento",
    "sistema_productivo"
]

def crear_preprocesador():
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERICAS),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore"),
                CATEGORICAS
            )
        ]
    )

# Comprobar dimensiones
print("=== MATRICES DE MODELADO ===")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"y_val:   {y_val.shape}")

print("\nVariables numéricas:", NUMERICAS)
print("Variables categóricas:", CATEGORICAS)

=== MATRICES DE MODELADO ===
X_train: (7334, 5)
y_train: (7334,)
X_val:   (1713, 5)
y_val:   (1713,)

Variables numéricas: ['semestre_b', 'rendimiento_rezago_1', 'rendimiento_rezago_2']
Variables categóricas: ['departamento', 'sistema_productivo']


In [6]:
def evaluar_cv_temporal(modelo, nombre_modelo):
    resultados = []

    for anios_entrenamiento, anio_validacion in VENTANAS_CV:

        # Separación temporal de cada ventana
        datos_train_cv = train[
            train["anio"].isin(anios_entrenamiento)
        ]

        datos_val_cv = train[
            train["anio"] == anio_validacion
        ]

        X_train_cv = datos_train_cv[VARIABLES]
        y_train_cv = datos_train_cv[OBJETIVO]

        X_val_cv = datos_val_cv[VARIABLES]
        y_val_cv = datos_val_cv[OBJETIVO]

        # Crear un pipeline independiente para cada ventana
        pipeline = Pipeline([
            ("preprocesador", crear_preprocesador()),
            ("modelo", modelo)
        ])

        pipeline.fit(X_train_cv, y_train_cv)

        predicciones = pipeline.predict(X_val_cv)

        metricas = calcular_metricas(
            y_val_cv,
            predicciones
        )

        resultados.append({
            "modelo": nombre_modelo,
            "anio_validacion": anio_validacion,
            "n_entrenamiento": len(datos_train_cv),
            "n_validacion": len(datos_val_cv),
            **metricas
        })

    return pd.DataFrame(resultados)

In [7]:
def evaluar_cv_temporal(modelo, nombre_modelo):
    resultados = []

    for anios_entrenamiento, anio_validacion in VENTANAS_CV:

        # Separación temporal de cada ventana
        datos_train_cv = train[
            train["anio"].isin(anios_entrenamiento)
        ]

        datos_val_cv = train[
            train["anio"] == anio_validacion
        ]

        X_train_cv = datos_train_cv[VARIABLES]
        y_train_cv = datos_train_cv[OBJETIVO]

        X_val_cv = datos_val_cv[VARIABLES]
        y_val_cv = datos_val_cv[OBJETIVO]

        # Crear un pipeline independiente para cada ventana
        pipeline = Pipeline([
            ("preprocesador", crear_preprocesador()),
            ("modelo", modelo)
        ])

        pipeline.fit(X_train_cv, y_train_cv)

        predicciones = pipeline.predict(X_val_cv)

        metricas = calcular_metricas(
            y_val_cv,
            predicciones
        )

        resultados.append({
            "modelo": nombre_modelo,
            "anio_validacion": anio_validacion,
            "n_entrenamiento": len(datos_train_cv),
            "n_validacion": len(datos_val_cv),
            **metricas
        })

    return pd.DataFrame(resultados)

In [8]:
# Random Forest con los hiperparámetros iniciales
params_rf_inicial = {
    "n_estimators": 200,
    "max_depth": None,
    "min_samples_leaf": 5,
}

rf_inicial = RandomForestRegressor(
    **params_rf_inicial,
    random_state=42,
    n_jobs=-1
)

# Evaluación mediante las tres ventanas temporales
resultados_rf_inicial = evaluar_cv_temporal(
    modelo=rf_inicial,
    nombre_modelo="RandomForest_inicial"
)

promedio_rf_inicial = resultados_rf_inicial[["rmse", "mae", "r2"]].mean()

# Registro en MLflow
with mlflow.start_run(
    run_name="RandomForest_inicial_t1_t2",
    tags={**TAGS_COMUNES, "tipo_run": "modelo_inicial", "modelo": "RandomForest"},
):
    mlflow.log_params({
        "modelo": "RandomForest",
        **params_rf_inicial,
        "random_state": 42,
        "tipo_validacion": "expanding_window",
        "numero_folds": len(VENTANAS_CV),
    })

    mlflow.log_metrics({
        "cv_rmse": float(promedio_rf_inicial["rmse"]),
        "cv_mae": float(promedio_rf_inicial["mae"]),
        "cv_r2": float(promedio_rf_inicial["r2"]),
        **{
            f"rmse_{int(a)}": float(r)
            for a, r in zip(resultados_rf_inicial["anio_validacion"], resultados_rf_inicial["rmse"])
        },
    })

print("=== RANDOM FOREST: VALIDACIÓN TEMPORAL ===")
display(resultados_rf_inicial.round(4))

print("\n=== PROMEDIO DE MÉTRICAS ===")
display(promedio_rf_inicial.round(4))

print("\n=== COMPARACIÓN DE RMSE CON PERSISTENCIA ===")

comparacion = pd.DataFrame({
    "anio": resultados_rf_inicial["anio_validacion"],
    "rmse_persistencia": df_baseline["rmse"],
    "rmse_random_forest": resultados_rf_inicial["rmse"]
})

comparacion["mejora_pct"] = (
    (comparacion["rmse_persistencia"] - comparacion["rmse_random_forest"])
    / comparacion["rmse_persistencia"]
    * 100
)

display(comparacion.round(4))

🏃 View run RandomForest_inicial_t1_t2 at: http://13.221.115.228:5000/#/experiments/2/runs/8eaa7ecce88549ef924dd36c063d415c
🧪 View experiment at: http://13.221.115.228:5000/#/experiments/2
=== RANDOM FOREST: VALIDACIÓN TEMPORAL ===


,modelo,anio_validacion,n_entrenamiento,n_validacion,rmse,mae,r2
0,RandomForest_inicial,2021,1839,1916,1.4893,0.7966,-0.0499
1,RandomForest_inicial,2022,3755,1810,0.6703,0.4128,0.7737
2,RandomForest_inicial,2023,5565,1769,0.5089,0.2929,0.8736



=== PROMEDIO DE MÉTRICAS ===


rmse    0.8895
mae     0.5008
r2      0.5325
dtype: float64


=== COMPARACIÓN DE RMSE CON PERSISTENCIA ===


,anio,rmse_persistencia,rmse_random_forest,mejora_pct
0,2021,1.6935,1.4893,12.0573
1,2022,0.6633,0.6703,-1.0536
2,2023,0.4985,0.5089,-2.0779


In [9]:
# Espacio inicial de búsqueda para Random Forest
parametros_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, None],
    "min_samples_leaf": [2, 5]
}

combinaciones_rf = list(ParameterGrid(parametros_rf))

print("=== BÚSQUEDA DE HIPERPARÁMETROS RANDOM FOREST ===")
print(f"Combinaciones: {len(combinaciones_rf)}")
print(f"Ventanas temporales: {len(VENTANAS_CV)}")
print(f"Entrenamientos previstos: {len(combinaciones_rf) * len(VENTANAS_CV)}")

display(pd.DataFrame(combinaciones_rf))

=== BÚSQUEDA DE HIPERPARÁMETROS RANDOM FOREST ===
Combinaciones: 12
Ventanas temporales: 3
Entrenamientos previstos: 36


,max_depth,min_samples_leaf,n_estimators
0,10.0,2,100
1,10.0,2,200
2,10.0,2,300
3,10.0,5,100
4,10.0,5,200
5,10.0,5,300
6,NaN,2,100
7,NaN,2,200
8,NaN,2,300
9,NaN,5,100


In [10]:
from time import perf_counter


def buscar_hiperparametros(nombre_modelo, clase_modelo, combinaciones, params_fijos):
    """Evalúa cada combinación con CV temporal y registra un run hijo por combinación
    dentro de un run padre. Devuelve la tabla ordenada y los mejores parámetros."""

    resultados = []
    inicio = perf_counter()

    with mlflow.start_run(
        run_name=f"{nombre_modelo}_busqueda_CV_t1_t2",
        tags={**TAGS_COMUNES, "tipo_run": "busqueda_padre", "modelo": nombre_modelo},
    ) as run_padre:

        mlflow.log_params({
            "modelo": nombre_modelo,
            "tipo_validacion": "expanding_window",
            "numero_folds": len(VENTANAS_CV),
            "numero_combinaciones": len(combinaciones),
            **{f"fijo_{k}": v for k, v in params_fijos.items()},
        })

        for i, params in enumerate(combinaciones, start=1):

            modelo = clase_modelo(**params, **params_fijos)
            resultados_cv = evaluar_cv_temporal(modelo=modelo, nombre_modelo=nombre_modelo)

            rmse_por_anio = {
                f"rmse_{int(a)}": float(r)
                for a, r in zip(resultados_cv["anio_validacion"], resultados_cv["rmse"])
            }
            metricas_cv = {
                "cv_rmse": float(resultados_cv["rmse"].mean()),
                "cv_mae": float(resultados_cv["mae"].mean()),
                "cv_r2": float(resultados_cv["r2"].mean()),
                **rmse_por_anio,
            }

            # Run hijo con la combinación evaluada
            with mlflow.start_run(
                run_name=f"{nombre_modelo}_{i:02d}",
                nested=True,
                tags={**TAGS_COMUNES, "tipo_run": "busqueda_hijo", "modelo": nombre_modelo},
            ) as run_hijo:
                mlflow.log_params({"modelo": nombre_modelo, **params})
                mlflow.log_metrics(metricas_cv)

            resultados.append({
                "params": params,
                "run_id": run_hijo.info.run_id,
                **params,
                **metricas_cv,
            })

            print(f"[{i:02d}/{len(combinaciones)}] cv_rmse: {metricas_cv['cv_rmse']:.4f}")

        # Mejor combinación según RMSE promedio
        mejor = min(resultados, key=lambda r: r["cv_rmse"])
        mejores_parametros = mejor["params"]

        df_busqueda = (
            pd.DataFrame(resultados)
            .drop(columns="params")
            .sort_values("cv_rmse")
            .reset_index(drop=True)
        )

        # Resumen en el run padre
        mlflow.log_params({f"mejor_{k}": v for k, v in mejores_parametros.items()})
        mlflow.log_metrics({k: mejor[k] for k in mejor if k.startswith(("cv_", "rmse_"))})
        mlflow.set_tag("mejor_run_hijo", mejor["run_id"])
        mlflow.log_table(
            df_busqueda,
            artifact_file=f"busqueda_hiperparametros_{nombre_modelo.lower()}.json",
        )

    print(f"\nTiempo total: {perf_counter() - inicio:.1f} s")
    print(f"Run padre: {run_padre.info.run_id}")
    print("Mejores parámetros:", mejores_parametros)

    return df_busqueda, mejores_parametros

In [11]:
df_busqueda_rf, mejores_parametros_rf = buscar_hiperparametros(
    nombre_modelo="RandomForest",
    clase_modelo=RandomForestRegressor,
    combinaciones=combinaciones_rf,
    params_fijos={"random_state": 42, "n_jobs": -1},
)

print("\n=== MEJORES COMBINACIONES RANDOM FOREST ===")
display(df_busqueda_rf.head(10).round(4))

🏃 View run RandomForest_01 at: http://13.221.115.228:5000/#/experiments/2/runs/91e9784b82d84375a48d7319a5b770e6
🧪 View experiment at: http://13.221.115.228:5000/#/experiments/2
[01/12] cv_rmse: 0.9102
🏃 View run RandomForest_02 at: http://13.221.115.228:5000/#/experiments/2/runs/6e7c2fa9ebcd4247b33966cd4e623eb2
🧪 View experiment at: http://13.221.115.228:5000/#/experiments/2
[02/12] cv_rmse: 0.9078
🏃 View run RandomForest_03 at: http://13.221.115.228:5000/#/experiments/2/runs/e18530e373cd475880e03dd50be2fa12
🧪 View experiment at: http://13.221.115.228:5000/#/experiments/2
[03/12] cv_rmse: 0.9072
🏃 View run RandomForest_04 at: http://13.221.115.228:5000/#/experiments/2/runs/e057cc1bf55845cf93f22d3c6ff508c5
🧪 View experiment at: http://13.221.115.228:5000/#/experiments/2
[04/12] cv_rmse: 0.8825
🏃 View run RandomForest_05 at: http://13.221.115.228:5000/#/experiments/2/runs/24254995431846a4af28f350b83eba2f
🧪 View experiment at: http://13.221.115.228:5000/#/experiments/2
[05/12] cv_rmse: 0.

,run_id,max_depth,min_samples_leaf,n_estimators,cv_rmse,cv_mae,cv_r2,rmse_2021,rmse_2022,rmse_2023
0,764c154d3be24e5dac01a6aa07c6ccd0,10.0,5,300,0.8813,0.4915,0.5381,1.4866,0.6573,0.5000
1,e057cc1bf55845cf93f22d3c6ff508c5,10.0,5,100,0.8825,0.4927,0.5374,1.4863,0.6607,0.5004
2,24254995431846a4af28f350b83eba2f,10.0,5,200,0.8825,0.4925,0.5368,1.4885,0.6592,0.4997
3,105258cd3e954ad481b467bb00dba521,NaN,5,300,0.8886,0.5001,0.5336,1.4874,0.6689,0.5094
4,bd1c9da2726b4a9b9b3ab38860d5033c,NaN,5,100,0.8894,0.5009,0.5331,1.4872,0.6713,0.5096
5,6f61ea6d8ab64921b3c035ae905fb502,NaN,5,200,0.8895,0.5008,0.5325,1.4893,0.6703,0.5089
6,e18530e373cd475880e03dd50be2fa12,10.0,2,300,0.9072,0.5005,0.5108,1.5301,0.6718,0.5198
7,6e7c2fa9ebcd4247b33966cd4e623eb2,10.0,2,200,0.9078,0.5011,0.5108,1.5289,0.6744,0.5200
8,91e9784b82d84375a48d7319a5b770e6,10.0,2,100,0.9102,0.5020,0.5081,1.5332,0.6763,0.5209
9,eebed1d7a2244ba7939212e193080f9d,NaN,2,300,0.9260,0.5167,0.4989,1.5313,0.7022,0.5445


In [12]:
# Espacio de búsqueda para XGBoost
parametros_xgb = {
    "n_estimators": [200, 400],
    "max_depth": [3, 5],
    "learning_rate": [0.03, 0.07],
    "subsample": [0.8, 1.0]
}

combinaciones_xgb = list(ParameterGrid(parametros_xgb))

print("=== BÚSQUEDA DE HIPERPARÁMETROS XGBOOST ===")
print(f"Combinaciones: {len(combinaciones_xgb)}")
print(f"Entrenamientos previstos: {len(combinaciones_xgb) * len(VENTANAS_CV)}\n")

df_busqueda_xgb, mejores_parametros_xgb = buscar_hiperparametros(
    nombre_modelo="XGBoost",
    clase_modelo=XGBRegressor,
    combinaciones=combinaciones_xgb,
    params_fijos={
        "colsample_bytree": 0.8,
        "objective": "reg:squarederror",
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": 0,
    },
)

print("\n=== MEJORES COMBINACIONES XGBOOST ===")
display(df_busqueda_xgb.head(10).round(4))

=== BÚSQUEDA DE HIPERPARÁMETROS XGBOOST ===
Combinaciones: 16
Entrenamientos previstos: 48

🏃 View run XGBoost_01 at: http://13.221.115.228:5000/#/experiments/2/runs/8523de41dd6646f6ad14ccd965ef100b
🧪 View experiment at: http://13.221.115.228:5000/#/experiments/2
[01/16] cv_rmse: 0.9048
🏃 View run XGBoost_02 at: http://13.221.115.228:5000/#/experiments/2/runs/1c0a78523cf44fa485fb278d479cc4d0
🧪 View experiment at: http://13.221.115.228:5000/#/experiments/2
[02/16] cv_rmse: 0.9068
🏃 View run XGBoost_03 at: http://13.221.115.228:5000/#/experiments/2/runs/ccbed7fd4ef84ac99814924466a29a59
🧪 View experiment at: http://13.221.115.228:5000/#/experiments/2
[03/16] cv_rmse: 0.9173
🏃 View run XGBoost_04 at: http://13.221.115.228:5000/#/experiments/2/runs/da4b4732b8c140f5b9fa396d5877be75
🧪 View experiment at: http://13.221.115.228:5000/#/experiments/2
[04/16] cv_rmse: 0.9141
🏃 View run XGBoost_05 at: http://13.221.115.228:5000/#/experiments/2/runs/f72a43a460f4404ba4823bf78ba268e1
🧪 View experiment

,run_id,learning_rate,max_depth,n_estimators,subsample,cv_rmse,cv_mae,cv_r2,rmse_2021,rmse_2022,rmse_2023
0,8523de41dd6646f6ad14ccd965ef100b,0.03,3,200,0.8,0.9048,0.5143,0.4919,1.6005,0.6127,0.5014
1,1c0a78523cf44fa485fb278d479cc4d0,0.03,3,200,1.0,0.9068,0.5181,0.4919,1.5967,0.6195,0.5042
2,da4b4732b8c140f5b9fa396d5877be75,0.03,3,400,1.0,0.9141,0.5180,0.4803,1.6196,0.6234,0.4995
3,ccbed7fd4ef84ac99814924466a29a59,0.03,3,400,0.8,0.9173,0.5147,0.4710,1.6430,0.6126,0.4963
4,2dea2195aa1049738f4b0519490bb876,0.07,3,200,0.8,0.9175,0.5142,0.4720,1.6396,0.6150,0.4977
5,852a946bfd6e4c6887455398c15d63b4,0.07,3,200,1.0,0.9234,0.5208,0.4673,1.6432,0.6263,0.5006
6,f72a43a460f4404ba4823bf78ba268e1,0.03,5,200,0.8,0.9264,0.5113,0.4601,1.6590,0.6279,0.4925
7,f6585a9a16504f4ea6994385df80c034,0.03,5,200,1.0,0.9286,0.5138,0.4569,1.6651,0.6262,0.4946
8,059b76d063e74409ab79a01aefb8dcb2,0.07,3,400,0.8,0.9366,0.5231,0.4474,1.6803,0.6268,0.5026
9,b21498d86859490eaa38b27e30bd60ed,0.07,3,400,1.0,0.9438,0.5286,0.4433,1.6793,0.6452,0.5069


In [13]:
# Construir modelos con los mejores hiperparámetros
modelos_optimizados = {
    "RandomForest_optimizado": RandomForestRegressor(
        **mejores_parametros_rf,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost_optimizado": XGBRegressor(
        **mejores_parametros_xgb,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )
}

resultados_validacion_2024 = []
pipelines_optimizados = {}

for nombre, modelo in modelos_optimizados.items():

    pipeline = Pipeline([
        ("preprocesador", crear_preprocesador()),
        ("modelo", modelo)
    ])

    # Entrenar exclusivamente con 2020-2023
    pipeline.fit(X_train, y_train)

    # Evaluar sobre 2024
    predicciones = pipeline.predict(X_val)
    metricas = calcular_metricas(y_val, predicciones)

    resultados_validacion_2024.append({
        "modelo": nombre,
        "n_validacion": len(y_val),
        **metricas
    })

    pipelines_optimizados[nombre] = pipeline

# Baseline de persistencia sobre las mismas observaciones
metricas_baseline_2024 = calcular_metricas(
    y_val,
    val["rendimiento_rezago_1"]
)

resultados_validacion_2024.append({
    "modelo": "Persistencia_t1",
    "n_validacion": len(y_val),
    **metricas_baseline_2024
})

df_validacion_2024 = (
    pd.DataFrame(resultados_validacion_2024)
    .sort_values("rmse")
    .reset_index(drop=True)
)

# Modelo seleccionado: el de menor RMSE entre los modelos entrenados
MODELO_SELECCIONADO = (
    df_validacion_2024[df_validacion_2024["modelo"] != "Persistencia_t1"]
    .iloc[0]["modelo"]
)

# Registro en MLflow
with mlflow.start_run(
    run_name="Comparacion_modelos_validacion_2024",
    tags={**TAGS_COMUNES, "tipo_run": "comparacion"},
):
    mlflow.log_params({
        "periodo_entrenamiento": "2020-2023",
        "periodo_validacion": "2024",
        "criterio_seleccion": "rmse",
        "modelo_seleccionado": MODELO_SELECCIONADO,
        **{f"rf_{k}": v for k, v in mejores_parametros_rf.items()},
        **{f"xgb_{k}": v for k, v in mejores_parametros_xgb.items()},
    })

    for _, fila in df_validacion_2024.iterrows():
        mlflow.log_metrics({
            f"{fila['modelo']}_rmse": float(fila["rmse"]),
            f"{fila['modelo']}_mae": float(fila["mae"]),
            f"{fila['modelo']}_r2": float(fila["r2"]),
        })

    mlflow.log_table(df_validacion_2024, artifact_file="comparacion_validacion_2024.json")

print("=== COMPARACIÓN DE MODELOS — VALIDACIÓN 2024 ===")
display(df_validacion_2024.round(4))

print("\nModelo seleccionado según RMSE:", MODELO_SELECCIONADO)

🏃 View run Comparacion_modelos_validacion_2024 at: http://13.221.115.228:5000/#/experiments/2/runs/4e3efa3b9ac14618860f00eb9a6c2ec5
🧪 View experiment at: http://13.221.115.228:5000/#/experiments/2
=== COMPARACIÓN DE MODELOS — VALIDACIÓN 2024 ===


,modelo,n_validacion,rmse,mae,r2
0,XGBoost_optimizado,1713,0.5220,0.2907,0.8603
1,RandomForest_optimizado,1713,0.5233,0.2723,0.8597
2,Persistencia_t1,1713,0.5672,0.2209,0.8351



Modelo seleccionado según RMSE: XGBoost_optimizado


In [14]:
assert MODELO_SELECCIONADO == "XGBoost_optimizado", (
    f"Se seleccionó {MODELO_SELECCIONADO}; ajustar las celdas finales"
)

# Entrenamiento final con datos de 2020 a 2024
datos_entrenamiento_final = df_modelo[
    df_modelo["anio"].between(2020, 2024)
].copy()

X_entrenamiento_final = datos_entrenamiento_final[VARIABLES]
y_entrenamiento_final = datos_entrenamiento_final[OBJETIVO]

X_test = test[VARIABLES]
y_test = test[OBJETIVO]

# Modelo seleccionado con los hiperparámetros óptimos
modelo_xgb_final = Pipeline([
    ("preprocesador", crear_preprocesador()),
    ("modelo", XGBRegressor(
        **mejores_parametros_xgb,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ))
])

modelo_xgb_final.fit(X_entrenamiento_final, y_entrenamiento_final)
predicciones_2025 = modelo_xgb_final.predict(X_test)

# Evaluación final y baseline sobre las mismas observaciones
metricas_xgb_2025 = calcular_metricas(y_test, predicciones_2025)
metricas_persistencia_2025 = calcular_metricas(y_test, test["rendimiento_rezago_1"])

df_evaluacion_2025 = pd.DataFrame([
    {"modelo": "XGBoost_optimizado", **metricas_xgb_2025},
    {"modelo": "Persistencia_t1", **metricas_persistencia_2025},
]).sort_values("rmse").reset_index(drop=True)

mejora_rmse = (
    (metricas_persistencia_2025["rmse"] - metricas_xgb_2025["rmse"])
    / metricas_persistencia_2025["rmse"]
    * 100
)

# Registro en MLflow
with mlflow.start_run(
    run_name="XGBoost_evaluacion_final_2025_t1_t2",
    tags={**TAGS_COMUNES, "tipo_run": "evaluacion_final", "modelo": "XGBoost"},
) as run_evaluacion:

    mlflow.log_params({
        "modelo": "XGBoost",
        **mejores_parametros_xgb,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "periodo_entrenamiento": "2020-2024",
        "periodo_prueba": "2025",
        "n_entrenamiento": len(X_entrenamiento_final),
        "n_prueba": len(X_test),
    })

    mlflow.log_metrics({
        "test_2025_rmse": float(metricas_xgb_2025["rmse"]),
        "test_2025_mae": float(metricas_xgb_2025["mae"]),
        "test_2025_r2": float(metricas_xgb_2025["r2"]),
        "baseline_2025_rmse": float(metricas_persistencia_2025["rmse"]),
        "baseline_2025_mae": float(metricas_persistencia_2025["mae"]),
        "baseline_2025_r2": float(metricas_persistencia_2025["r2"]),
        "mejora_rmse_pct": float(mejora_rmse),
    })

    mlflow.log_table(df_evaluacion_2025, artifact_file="evaluacion_final_2025.json")

RUN_ID_EVALUACION = run_evaluacion.info.run_id

print("=== EVALUACIÓN FINAL — 2025 ===")
print(f"Observaciones de entrenamiento: {len(X_entrenamiento_final):,}")
print(f"Observaciones de prueba: {len(X_test):,}")
display(df_evaluacion_2025.round(4))
print(f"\nMejora de RMSE frente a persistencia: {mejora_rmse:.2f}%")

🏃 View run XGBoost_evaluacion_final_2025_t1_t2 at: http://13.221.115.228:5000/#/experiments/2/runs/b5caadcb80ce4be79a7067e17cb073d3
🧪 View experiment at: http://13.221.115.228:5000/#/experiments/2
=== EVALUACIÓN FINAL — 2025 ===
Observaciones de entrenamiento: 9,047
Observaciones de prueba: 1,735


,modelo,rmse,mae,r2
0,XGBoost_optimizado,0.3677,0.2139,0.9287
1,Persistencia_t1,0.4140,0.1478,0.9097



Mejora de RMSE frente a persistencia: 11.17%


In [15]:
# Preparación del conjunto de entrenamiento para producción
datos_produccion = df_modelo[
    df_modelo["anio"].between(2020, 2025)
].copy()

X_produccion = datos_produccion[VARIABLES]
y_produccion = datos_produccion[OBJETIVO]

print("=== DATOS PARA MODELO DE PRODUCCIÓN ===")
print(f"Observaciones: {len(datos_produccion):,}")
print(f"Período: {datos_produccion['anio'].min()}–{datos_produccion['anio'].max()}")

print("\n=== VALORES FALTANTES ===")
print(X_produccion.isna().sum())

print("\n=== DISTRIBUCIÓN POR AÑO ===")
print(datos_produccion.groupby("anio").size())

print("\n=== HIPERPARÁMETROS SELECCIONADOS ===")
print(mejores_parametros_xgb)

=== DATOS PARA MODELO DE PRODUCCIÓN ===
Observaciones: 10,782
Período: 2020–2025

=== VALORES FALTANTES ===
semestre_b              0
rendimiento_rezago_1    0
rendimiento_rezago_2    0
departamento            0
sistema_productivo      0
dtype: int64

=== DISTRIBUCIÓN POR AÑO ===
anio
2020    1839
2021    1916
2022    1810
2023    1769
2024    1713
2025    1735
dtype: int64

=== HIPERPARÁMETROS SELECCIONADOS ===
{'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}


In [16]:
from time import perf_counter

# Pipeline definitivo de producción
modelo_xgb_produccion = Pipeline([
    ("preprocesador", crear_preprocesador()),
    ("modelo", XGBRegressor(
        **mejores_parametros_xgb,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ))
])

# Entrenamiento con todos los datos disponibles
inicio = perf_counter()

modelo_xgb_produccion.fit(
    X_produccion,
    y_produccion
)

tiempo_entrenamiento = perf_counter() - inicio

print("=== MODELO DE PRODUCCIÓN ===")
print("Entrenamiento completado correctamente.")
print(f"Observaciones utilizadas: {len(X_produccion):,}")
print(f"Variables predictoras: {len(VARIABLES)}")
print(f"Tiempo de entrenamiento: {tiempo_entrenamiento:.2f} segundos")

print("\n=== CONFIGURACIÓN DEL MODELO ===")
print(modelo_xgb_produccion.named_steps["modelo"].get_params())

=== MODELO DE PRODUCCIÓN ===
Entrenamiento completado correctamente.
Observaciones utilizadas: 10,782
Variables predictoras: 5
Tiempo de entrenamiento: 0.29 segundos

=== CONFIGURACIÓN DEL MODELO ===
{'objective': 'reg:squarederror', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.8, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': None, 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.03, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 3, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 200, 'n_jobs': -1, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None

In [18]:
import skops.io as sio
from tempfile import TemporaryDirectory
from mlflow.models import infer_signature

NOMBRE_MODELO_REGISTRO = "agropredict-maiz-xgb"

# Tipos de XGBoost que skops debe aceptar al serializar y cargar el pipeline
TIPOS_CONFIABLES = [
    "xgboost.core.Booster",
    "xgboost.sklearn.XGBRegressor",
]

# Firma y ejemplo de entrada para documentar el contrato del modelo
firma = infer_signature(X_produccion, modelo_xgb_produccion.predict(X_produccion))
ejemplo_entrada = X_produccion.head(5)

with mlflow.start_run(
    run_name="XGBoost_modelo_produccion_t1_t2",
    tags={
        **TAGS_COMUNES,
        "tipo_run": "produccion",
        "modelo": "XGBoost",
        "run_evaluacion_2025": RUN_ID_EVALUACION,
    },
) as run_produccion:

    mlflow.log_params({
        "modelo": "XGBoost",
        **mejores_parametros_xgb,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "periodo_entrenamiento": "2020-2025",
        "observaciones_entrenamiento": len(X_produccion),
        "variables": ",".join(VARIABLES),
    })

    # Tiempo de entrenamiento y métricas de referencia de la evaluación 2025
    mlflow.log_metrics({
        "tiempo_entrenamiento_segundos": tiempo_entrenamiento,
        "ref_test_2025_rmse": float(metricas_xgb_2025["rmse"]),
        "ref_test_2025_mae": float(metricas_xgb_2025["mae"]),
        "ref_test_2025_r2": float(metricas_xgb_2025["r2"]),
    })

    # Modelo en formato MLflow, registrado en el Model Registry
    info_modelo = mlflow.sklearn.log_model(
        sk_model=modelo_xgb_produccion,
        name="modelo",
        signature=firma,
        input_example=ejemplo_entrada,
        registered_model_name=NOMBRE_MODELO_REGISTRO,
        skops_trusted_types=TIPOS_CONFIABLES,
    )

    # Copia en formato skops, la que usa la API
    with TemporaryDirectory() as directorio:
        ruta_skops = Path(directorio) / "xgboost_maiz_t1_t2.skops"
        sio.dump(modelo_xgb_produccion, ruta_skops)
        mlflow.log_artifact(str(ruta_skops), artifact_path="modelo_skops")

RUN_ID_PRODUCCION = run_produccion.info.run_id

# Versión registrada y alias champion
cliente = mlflow.MlflowClient()
version_registrada = cliente.search_model_versions(
    f"name='{NOMBRE_MODELO_REGISTRO}' and run_id='{RUN_ID_PRODUCCION}'"
)[0].version

cliente.set_registered_model_alias(NOMBRE_MODELO_REGISTRO, "champion", version_registrada)
cliente.update_model_version(
    name=NOMBRE_MODELO_REGISTRO,
    version=version_registrada,
    description=(
        f"XGBoost con rezagos t-1 y t-2, entrenado con 2020-2025. "
        f"Test 2025: RMSE {metricas_xgb_2025['rmse']:.4f}, "
        f"MAE {metricas_xgb_2025['mae']:.4f}, R2 {metricas_xgb_2025['r2']:.4f}."
    ),
)

print("=== MODELO REGISTRADO EN MLFLOW ===")
print(f"Run ID: {RUN_ID_PRODUCCION}")
print(f"Modelo registrado: {NOMBRE_MODELO_REGISTRO} v{version_registrada} (alias: champion)")
print("Artefactos: modelo/ (MLflow) y modelo_skops/xgboost_maiz_t1_t2.skops")

/Users/Simon/Library/CloudStorage/OneDrive-SharedLibraries-ProductosAlimenticiosPascualS.A/Inteligencia Comercial - 5-Semestre/MIID-4304 Despliegue/Proyecto/agropredict-colombia/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Successfully registered model 'agropredict-maiz-xgb'.
2026/11/21 20:45:09 I

🏃 View run XGBoost_modelo_produccion_t1_t2 at: http://13.221.115.228:5000/#/experiments/2/runs/5694caa8bcbb4eb4a038ef0cfc0613d2
🧪 View experiment at: http://13.221.115.228:5000/#/experiments/2
=== MODELO REGISTRADO EN MLFLOW ===
Run ID: 5694caa8bcbb4eb4a038ef0cfc0613d2
Modelo registrado: agropredict-maiz-xgb v1 (alias: champion)
Artefactos: modelo/ (MLflow) y modelo_skops/xgboost_maiz_t1_t2.skops


In [19]:
from mlflow.artifacts import download_artifacts

# 1) Modelo desde el Model Registry (alias champion)
uri_champion = f"models:/{NOMBRE_MODELO_REGISTRO}@champion"
modelo_registry = mlflow.sklearn.load_model(uri_champion)

# 2) Copia .skops del run de producción (la que usará la API)
ruta_recuperada = download_artifacts(
    run_id=RUN_ID_PRODUCCION,
    artifact_path="modelo_skops/xgboost_maiz_t1_t2.skops"
)

tipos_no_confiables = sio.get_untrusted_types(file=ruta_recuperada)
assert set(tipos_no_confiables) == set(TIPOS_CONFIABLES), (
    f"Se encontraron tipos inesperados: {tipos_no_confiables}"
)

modelo_skops = sio.load(ruta_recuperada, trusted=TIPOS_CONFIABLES)

# 3) Comparar contra el modelo en memoria sobre todo el conjunto de producción
pred_original = modelo_xgb_produccion.predict(X_produccion)
pred_registry = modelo_registry.predict(X_produccion)
pred_skops = modelo_skops.predict(X_produccion)

coincide_registry = np.allclose(pred_original, pred_registry, rtol=1e-6, atol=1e-8)
coincide_skops = np.allclose(pred_original, pred_skops, rtol=1e-6, atol=1e-8)

print("=== VERIFICACIÓN DEL MODELO RECUPERADO ===")
print(f"Registry: {uri_champion} (v{version_registrada})")
print(f"Archivo skops: {ruta_recuperada}")
print(f"Tipos autorizados: {tipos_no_confiables}")
print(f"Coincide registry vs memoria: {coincide_registry} "
      f"(dif. máx. {np.max(np.abs(pred_original - pred_registry)):.2e})")
print(f"Coincide skops vs memoria:    {coincide_skops} "
      f"(dif. máx. {np.max(np.abs(pred_original - pred_skops)):.2e})")

display(pd.DataFrame({
    "prediccion_original": pred_original[:10],
    "prediccion_registry": pred_registry[:10],
    "prediccion_skops": pred_skops[:10],
}).round(6))

assert coincide_registry and coincide_skops, "Las predicciones no coinciden."

=== VERIFICACIÓN DEL MODELO RECUPERADO ===
Registry: models:/agropredict-maiz-xgb@champion (v1)
Archivo skops: /var/folders/lm/21qrb3m15fd4zm5fypxtqvf80000gp/T/tmpkqzjcik_/xgboost_maiz_t1_t2.skops
Tipos autorizados: ['xgboost.core.Booster', 'xgboost.sklearn.XGBRegressor']
Coincide registry vs memoria: True (dif. máx. 0.00e+00)
Coincide skops vs memoria:    True (dif. máx. 0.00e+00)


,prediccion_original,prediccion_registry,prediccion_skops
0,1.965036,1.965036,1.965036
1,1.965484,1.965484,1.965484
2,1.965036,1.965036,1.965036
3,1.965484,1.965484,1.965484
4,1.965036,1.965036,1.965036
5,1.965484,1.965484,1.965484
6,1.965036,1.965036,1.965036
7,1.965484,1.965484,1.965484
8,1.965036,1.965036,1.965036
9,1.965484,1.965484,1.965484


In [20]:
import json
import shutil

DIR_MODELOS = PROYECTO / "models"
ruta_destino = DIR_MODELOS / "xgboost_maiz_t1_t2.skops"
ruta_metadatos = DIR_MODELOS / "xgboost_maiz_t1_t2.json"

# Solo se copia si la verificación anterior pasó
assert coincide_registry and coincide_skops, "Verificación fallida: no se copia el modelo."

# Sobrescribe el modelo anterior (la versión previa queda en el historial de DVC)
shutil.copy2(ruta_recuperada, ruta_destino)

# Metadatos del modelo para trazabilidad (paquete y API)
metadatos = {
    "nombre_registry": NOMBRE_MODELO_REGISTRO,
    "version_registry": int(version_registrada),
    "alias": "champion",
    "run_id_produccion": RUN_ID_PRODUCCION,
    "run_id_evaluacion_2025": RUN_ID_EVALUACION,
    "experimento": MLFLOW_EXPERIMENTO,
    "variables": VARIABLES,
    "categoricas": CATEGORICAS,
    "numericas": NUMERICAS,
    "departamentos": sorted(X_produccion["departamento"].unique().tolist()),
    "sistemas_productivos": sorted(X_produccion["sistema_productivo"].unique().tolist()),
    "hiperparametros": {**mejores_parametros_xgb, "colsample_bytree": 0.8, "random_state": 42},
    "periodo_entrenamiento": "2020-2025",
    "observaciones_entrenamiento": len(X_produccion),
    "metricas_test_2025": {k: round(float(v), 4) for k, v in metricas_xgb_2025.items()},
    "metricas_persistencia_2025": {k: round(float(v), 4) for k, v in metricas_persistencia_2025.items()},
    "tipos_skops_confiables": TIPOS_CONFIABLES,
}

ruta_metadatos.write_text(json.dumps(metadatos, indent=2, ensure_ascii=False), encoding="utf-8")

print("=== MODELO PREPARADO PARA VERSIONAMIENTO ===")
print("Modelo:", ruta_destino, f"({ruta_destino.stat().st_size / 1024:.1f} KB)")
print("Metadatos:", ruta_metadatos)
print(json.dumps(metadatos["metricas_test_2025"], indent=2))

=== MODELO PREPARADO PARA VERSIONAMIENTO ===
Modelo: /Users/Simon/Library/CloudStorage/OneDrive-SharedLibraries-ProductosAlimenticiosPascualS.A/Inteligencia Comercial - 5-Semestre/MIID-4304 Despliegue/Proyecto/agropredict-colombia/models/xgboost_maiz_t1_t2.skops (351.8 KB)
Metadatos: /Users/Simon/Library/CloudStorage/OneDrive-SharedLibraries-ProductosAlimenticiosPascualS.A/Inteligencia Comercial - 5-Semestre/MIID-4304 Despliegue/Proyecto/agropredict-colombia/models/xgboost_maiz_t1_t2.json
{
  "rmse": 0.3677,
  "mae": 0.2139,
  "r2": 0.9287
}
